# Metafeatures Relevance Testing

In [1]:
%pip install --upgrade jupyter ipywidgets --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
DATABASE_PATH = "/mnt/sata1/lam/EMBER2024/TRAIN.sqlite3"
NUM_FEATURES = 2568

In [3]:
import sqlite3
import numpy as np

class MyDataLoader:
    def __init__(self) -> None:
        self.conn = sqlite3.connect(DATABASE_PATH)
        self.cursor = self.conn.cursor()
        self.cursor.execute("SELECT sha256, label, feature_vector FROM files ORDER BY sha256;")

    def load_next(self, count: int) -> tuple[np.ndarray, np.ndarray]:
        """Returns X and y for the NEXT `count` samples from the database."""

        X = []
        y = []
        for row in self.cursor.fetchmany(count):
            sha256, label, feature_vector = row
            # print(f"SHA256: {sha256}, Label: {label}, Feature Vector Length: {len(feature_vector)}")
            feature_vector = np.frombuffer(feature_vector, dtype=np.float32)
            X0 = feature_vector
            y0 = label
            X.append(X0)
            y.append(y0)
        
        return np.array(X), np.array(y)

loader = MyDataLoader()

In [4]:
X_train, y_train = loader.load_next(131072)
X_val, y_val = loader.load_next(16384)
X_test, y_test = loader.load_next(16384)

In [5]:
from utils.metafeature_experiment import MetafeatureExperiment
from utils.feature_vector_transformer import FeatureAddition, FeatureRemoval

def run_experiment(alterations: list):
    experiment = MetafeatureExperiment(
        alterations=alterations
    )
    result = experiment.run(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test,
    )
    print(str(result))
    return result


In [13]:

baseline_result = run_experiment(alterations=[])

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.996538508857
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [7]:

experiment1_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'GeneralFileInfo;start_bytes;0',
            'GeneralFileInfo;start_bytes;1',
            'GeneralFileInfo;start_bytes;2',
            'GeneralFileInfo;start_bytes;3',

            'HeaderFileInfo;COFF;timestamp',
            'HeaderFileInfo;COFF;pointer_to_symbol_table', #
            'HeaderFileInfo;COFF;machine',
            'HeaderFileInfo;COFF;has_characteristics;DLL',
            'HeaderFileInfo;OPTIONAL;subsystem',
            'HeaderFileInfo;OPTIONAL;major_image_version',
            'HeaderFileInfo;OPTIONAL;minor_image_version',
            'HeaderFileInfo;OPTIONAL;major_linker_version',
            'HeaderFileInfo;OPTIONAL;minor_linker_version',
            'HeaderFileInfo;OPTIONAL;major_operating_system_version',
            'HeaderFileInfo;OPTIONAL;minor_operating_system_version',
            'HeaderFileInfo;OPTIONAL;major_subsystem_version',
            'HeaderFileInfo;OPTIONAL;minor_subsystem_version',
            'HeaderFileInfo;OPTIONAL;address_of_entrypoint', #
            'HeaderFileInfo;OPTIONAL;base_of_code', #
            'HeaderFileInfo;OPTIONAL;image_base', #
        ]),
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.9965
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [8]:
experiment2_result = run_experiment([
    *(FeatureRemoval(x) for x in [
        'HeaderFileInfo;COFF;timestamp',
    ]),
])

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.9965
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [9]:
experiment3_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'HeaderFileInfo;COFF;timestamp',
            # 'HeaderFileInfo;COFF;pointer_to_symbol_table',
            # 'HeaderFileInfo;OPTIONAL;address_of_entrypoint',
            # 'HeaderFileInfo;OPTIONAL;base_of_code',
            # 'HeaderFileInfo;OPTIONAL;image_base',
            'HeaderFileInfo;COFF;machine',
            'HeaderFileInfo;COFF;has_characteristics;DLL',
            'HeaderFileInfo;OPTIONAL;subsystem',
            'HeaderFileInfo;OPTIONAL;major_image_version',
            'HeaderFileInfo;OPTIONAL;minor_image_version',
            'HeaderFileInfo;OPTIONAL;major_linker_version',
            'HeaderFileInfo;OPTIONAL;minor_linker_version',
            'HeaderFileInfo;OPTIONAL;major_operating_system_version',
            'HeaderFileInfo;OPTIONAL;minor_operating_system_version',
            'HeaderFileInfo;OPTIONAL;major_subsystem_version',
            'HeaderFileInfo;OPTIONAL;minor_subsystem_version',
        ])
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.9966
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [10]:
experiment4_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'HeaderFileInfo;COFF;timestamp',
            'HeaderFileInfo;COFF;pointer_to_symbol_table',
            'HeaderFileInfo;OPTIONAL;address_of_entrypoint',
            'HeaderFileInfo;OPTIONAL;base_of_code',
            'HeaderFileInfo;OPTIONAL;image_base',
        ])
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.9964
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [11]:
experiment5_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'HeaderFileInfo;COFF;pointer_to_symbol_table',
            'HeaderFileInfo;OPTIONAL;address_of_entrypoint',
            'HeaderFileInfo;OPTIONAL;base_of_code',
            'HeaderFileInfo;OPTIONAL;image_base',
        ])
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.9965
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [12]:
[r.auc for r in [
    baseline_result,
    experiment1_result,
    experiment2_result,
    experiment3_result,
    experiment4_result,
    experiment5_result,
]]

[0.9965385088569028,
 0.9964773797700667,
 0.996548847974059,
 0.9965673527297285,
 0.9963852786125671,
 0.9964527621057138]

## Observations

Experiments 2 and 3 surpassed the baseline in AUC by a small margin.

Removing `HeaderFileInfo;COFF;timestamp` alone (experiment 2) increased AUC by 0.00001.

**Further** removing the following features (experiment 3) increased AUC by 0.00002.

- `HeaderFileInfo;COFF;machine`
- `HeaderFileInfo;COFF;has_characteristics;DLL`
- `HeaderFileInfo;OPTIONAL;subsystem`
- `HeaderFileInfo;OPTIONAL;major_image_version`
- `HeaderFileInfo;OPTIONAL;minor_image_version`
- `HeaderFileInfo;OPTIONAL;major_linker_version`
- `HeaderFileInfo;OPTIONAL;minor_linker_version`
- `HeaderFileInfo;OPTIONAL;major_operating_system_version`
- `HeaderFileInfo;OPTIONAL;minor_operating_system_version`
- `HeaderFileInfo;OPTIONAL;major_subsystem_version`
- `HeaderFileInfo;OPTIONAL;minor_subsystem_version`

The following features seem to be important, as their removal (experiment 4) caused a significant drop in AUC by 0.0001.

- `HeaderFileInfo;COFF;pointer_to_symbol_table`
- `HeaderFileInfo;OPTIONAL;address_of_entrypoint`
- `HeaderFileInfo;OPTIONAL;base_of_code`
- `HeaderFileInfo;OPTIONAL;image_base`


In [7]:
experiment6_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'GeneralFileInfo;size',
            'GeneralFileInfo;start_bytes;0',
            'GeneralFileInfo;start_bytes;1',
            'GeneralFileInfo;start_bytes;2',
            'GeneralFileInfo;start_bytes;3',
        ])
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.9966
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [11]:
experiment6_result.auc

0.9966387152785089

## Observations

Removing the file size and the first bytes of the file from the feature vector also yielded a small improvement in AUC, this time greater than before - by 0.0001.

In [7]:
experiment7_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            # from experiment 6
            'GeneralFileInfo;size',
            'GeneralFileInfo;start_bytes;0',
            'GeneralFileInfo;start_bytes;1',
            'GeneralFileInfo;start_bytes;2',
            'GeneralFileInfo;start_bytes;3',

            # from experiment 3
            'HeaderFileInfo;COFF;timestamp',
            'HeaderFileInfo;COFF;machine',
            'HeaderFileInfo;COFF;has_characteristics;DLL',
            'HeaderFileInfo;OPTIONAL;subsystem',
            'HeaderFileInfo;OPTIONAL;major_image_version',
            'HeaderFileInfo;OPTIONAL;minor_image_version',
            'HeaderFileInfo;OPTIONAL;major_linker_version',
            'HeaderFileInfo;OPTIONAL;minor_linker_version',
            'HeaderFileInfo;OPTIONAL;major_operating_system_version',
            'HeaderFileInfo;OPTIONAL;minor_operating_system_version',
            'HeaderFileInfo;OPTIONAL;major_subsystem_version',
            'HeaderFileInfo;OPTIONAL;minor_subsystem_version',
        ]),
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.996454573338
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [8]:
experiment8_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            # from experiment 6
            'GeneralFileInfo;size',
            'GeneralFileInfo;start_bytes;0',
            'GeneralFileInfo;start_bytes;1',
            'GeneralFileInfo;start_bytes;2',
            'GeneralFileInfo;start_bytes;3',

            # from experiment 2
            'HeaderFileInfo;COFF;timestamp',
        ]),
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.996400885397
New Feature Impacts:
  (no new features)


/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## Observations

Strangely, combining removal approaches of the last most successful experiments drops AUC by 0.0002.

Let us try some new features.

In [6]:
from utils.feature_vector_transformer import FeatureVectorView

class SizeofCodePerSizeofImage(FeatureAddition):
    def __init__(self) -> None:
        super().__init__("SizeofCodePerSizeofImage")
    
    def apply(self, vector_view: FeatureVectorView) -> float:
        sizeof_code = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_code')
        sizeof_image = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_image')
        if sizeof_image == 0:
            return 0.0
        return sizeof_code / sizeof_image


In [7]:
class SizeofInitializedDataPerSizeofImage(FeatureAddition):
    def __init__(self) -> None:
        super().__init__("SizeofInitializedDataPerSizeofImage")
    
    def apply(self, vector_view: FeatureVectorView) -> float:
        sizeof_initialized_data = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_initialized_data')
        sizeof_image = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_image')
        if sizeof_image == 0:
            return 0.0
        return sizeof_initialized_data / sizeof_image

In [8]:
class SizeofUninitializedDataPerSizeofImage(FeatureAddition):
    def __init__(self) -> None:
        super().__init__("SizeofUninitializedDataPerSizeofImage")
    
    def apply(self, vector_view: FeatureVectorView) -> float:
        sizeof_uninitialized_data = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_uninitialized_data')
        sizeof_image = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_image')
        if sizeof_image == 0:
            return 0.0
        return sizeof_uninitialized_data / sizeof_image

In [9]:
class SizeofHeadersPerSizeofImage(FeatureAddition):
    def __init__(self) -> None:
        super().__init__("SizeofHeadersPerSizeofImage")
    
    def apply(self, vector_view: FeatureVectorView) -> float:
        sizeof_headers = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_headers')
        sizeof_image = vector_view.get('HeaderFileInfo;OPTIONAL;sizeof_image')
        if sizeof_image == 0:
            return 0.0
        return sizeof_headers / sizeof_image

In [10]:
class MaxSectionVsizeAndSizeRatiosDelta(FeatureAddition):
    def __init__(self) -> None:
        super().__init__("MaxSectionVsizeAndSizeRatiosDelta")
    
    def apply(self, vector_view: FeatureVectorView) -> float:
        max_vsize = vector_view.get('SectionInfo;general;section_vsize_ratio;max')
        max_size = vector_view.get('SectionInfo;general;section_size_ratio;max')
        return max_vsize - max_size


In [11]:
experiment9_result = run_experiment(
    alterations=[
        SizeofCodePerSizeofImage(),
        SizeofInitializedDataPerSizeofImage(),
        SizeofUninitializedDataPerSizeofImage(),
        SizeofHeadersPerSizeofImage(),
        MaxSectionVsizeAndSizeRatiosDelta(),
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.996653054200
New Feature Impacts:
  - SizeofCodePerSizeofImage: E(|SHAP|)=0.050951, Ranking=30, Percentile=98.83%, Delta from Avg=0.046232, Delta from Max=-1.620882, Delta from Min=0.050951
  - SizeofInitializedDataPerSizeofImage: E(|SHAP|)=0.025319, Ranking=67, Percentile=97.40%, Delta from Avg=0.020601, Delta from Max=-1.646513, Delta from Min=0.025319
  - SizeofUninitializedDataPerSizeofImage: E(|SHAP|)=0.005427, Ranking=310, Percenti

/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## Observations

Adding a few statistical features based on existing features seems to help a bit.

Now remove `COFF;timestamp` like in experiment 2 above.

In [12]:
experiment10_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            # from experiment 2
            'HeaderFileInfo;COFF;timestamp',
        ]),
        
        SizeofCodePerSizeofImage(),
        SizeofInitializedDataPerSizeofImage(),
        SizeofUninitializedDataPerSizeofImage(),
        SizeofHeadersPerSizeofImage(),
        MaxSectionVsizeAndSizeRatiosDelta(),
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.996480519239
New Feature Impacts:
  - SizeofCodePerSizeofImage: E(|SHAP|)=0.057545, Ranking=26, Percentile=98.99%, Delta from Avg=0.052817, Delta from Max=-1.605471, Delta from Min=0.057545
  - SizeofInitializedDataPerSizeofImage: E(|SHAP|)=0.016823, Ranking=95, Percentile=96.31%, Delta from Avg=0.012094, Delta from Max=-1.646194, Delta from Min=0.016823
  - SizeofUninitializedDataPerSizeofImage: E(|SHAP|)=0.006527, Ranking=275, Percenti

/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## Observations

Even weirder, removing useless feature `COFF;timestamp` has devastating effect - while it has been shown that this particular feature does not help much. Let us try combining these new features with experiment 3's removal strategy.

In [11]:
experiment11_result = run_experiment(
    alterations=[
        *(FeatureRemoval(x) for x in [
            'HeaderFileInfo;COFF;timestamp',
            # 'HeaderFileInfo;COFF;pointer_to_symbol_table',
            # 'HeaderFileInfo;OPTIONAL;address_of_entrypoint',
            # 'HeaderFileInfo;OPTIONAL;base_of_code',
            # 'HeaderFileInfo;OPTIONAL;image_base',
            'HeaderFileInfo;COFF;machine',
            'HeaderFileInfo;COFF;has_characteristics;DLL',
            'HeaderFileInfo;OPTIONAL;subsystem',
            'HeaderFileInfo;OPTIONAL;major_image_version',
            'HeaderFileInfo;OPTIONAL;minor_image_version',
            'HeaderFileInfo;OPTIONAL;major_linker_version',
            'HeaderFileInfo;OPTIONAL;minor_linker_version',
            'HeaderFileInfo;OPTIONAL;major_operating_system_version',
            'HeaderFileInfo;OPTIONAL;minor_operating_system_version',
            'HeaderFileInfo;OPTIONAL;major_subsystem_version',
            'HeaderFileInfo;OPTIONAL;minor_subsystem_version',
        ]),
        
        SizeofCodePerSizeofImage(),
        SizeofInitializedDataPerSizeofImage(),
        SizeofUninitializedDataPerSizeofImage(),
        SizeofHeadersPerSizeofImage(),
        MaxSectionVsizeAndSizeRatiosDelta(),
    ]
)

[MetafeatureExperiment] Transforming training set...
[MetafeatureExperiment] Transforming validation set...
[MetafeatureExperiment] Training model...
[MetafeatureExperiment] Transforming test set...
[MetafeatureExperiment] Running model predictions on test set...
[MetafeatureExperiment] Computing SHAP values on test set...
[MetafeatureExperiment] Computing global SHAP values (for test set)...
[MetafeatureExperiment] Computing AUC on test set...
[MetafeatureExperiment] Computing new feature impacts...
[MetafeatureExperiment] Experiment completed.
AUC: 0.996667664807
New Feature Impacts:
  - SizeofCodePerSizeofImage: E(|SHAP|)=0.048738, Ranking=40, Percentile=98.44%, Delta from Avg=0.044031, Delta from Max=-0.909336, Delta from Min=0.048738
  - SizeofInitializedDataPerSizeofImage: E(|SHAP|)=0.015920, Ranking=110, Percentile=95.70%, Delta from Avg=0.011213, Delta from Max=-0.942154, Delta from Min=0.015920
  - SizeofUninitializedDataPerSizeofImage: E(|SHAP|)=0.006319, Ranking=265, Percent

/home/lam/Desktop/Viettel/EMBER2024/venv/lib/python3.12/site-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## Observations

Okay, the gain is now noticeable.

In [14]:
gain = experiment11_result.auc - baseline_result.auc
gain

0.0001291559496435779

# Good bye